# Japan Road Traffic Volume: a first look

Hourly vehicle counts from about 2,000 traffic counters on Japan's national highways
(MLIT Traffic Volume API, reference values, data provided by JARTIC). This notebook loads the
files, shows when traffic peaks, and asks a simple question: **was a road busier than usual?**

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# On Kaggle the files sit somewhere under /kaggle/input; elsewhere they are in data/.
found = sorted(glob.glob("/kaggle/input/**/counters.csv", recursive=True)) + ["data/counters.csv"]
DATA = os.path.dirname(next(p for p in found if os.path.exists(p)))
plt.rcParams.update({"axes.titlesize": 16, "axes.labelsize": 14, "legend.fontsize": 12,
                     "xtick.labelsize": 12, "ytick.labelsize": 12})

counters = pd.read_csv(f"{DATA}/counters.csv", dtype={"municipality_code": str})
hourly = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f"{DATA}/hourly_*.parquet"))],
                   ignore_index=True)
print(f"{len(hourly):,} hourly rows, {hourly.time_jst.min()} to {hourly.time_jst.max()}")
print(hourly.groupby("sensor").counter_id.nunique().rename("counters"))

## One number per row: vehicles in both directions
Permanent counters (`loop`) give small, large and unclassified vehicles; CCTV counters give the
total directly. CCTV counts are often empty (camera off its preset position and other reasons),
so this notebook uses permanent counters only. A permanent counter that fails reports **0 vehicles
with a fault or missing flag**, so flagged rows are dropped first.

In [ ]:
loop = hourly[hourly.sensor == "loop"].copy()
flags = [f"{d}_{k}" for d in ("up", "down") for k in ("power_failure", "loop_fault", "ultrasonic_fault", "missing")]
flagged = loop[flags].fillna(False).any(axis=1)
print(f"rows flagged by the source: {flagged.mean():.1%} (dropped)")
loop = loop[~flagged]
parts = ["up_small", "up_large", "up_unclassified", "down_small", "down_large", "down_unclassified"]
loop[parts] = loop[parts].astype("float64")  # nullable integers -> float, empty -> NaN
loop["vehicles"] = loop[parts].sum(axis=1, min_count=6)  # NaN unless all six are present
loop["share_large"] = loop[["up_large", "down_large"]].sum(axis=1) / loop.vehicles
print(f"rows with a full count: {loop.vehicles.notna().mean():.1%}")

## When is traffic heaviest?

In [ ]:
loop["hour"] = loop.time_jst.dt.hour
loop["weekday"] = loop.time_jst.dt.dayofweek
grid = loop.groupby(["weekday", "hour"]).vehicles.mean().unstack()

fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(grid, aspect="auto", cmap="magma")
ax.set_yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour (JST)")
ax.set_title("Mean vehicles per hour at a permanent counter")
fig.colorbar(im, ax=ax, label="vehicles / hour")
plt.tight_layout()
plt.show()

## Trucks at night
The share of large vehicles rises at night, when freight dominates.

In [ ]:
share = loop.groupby("hour").apply(lambda g: g[["up_large", "down_large"]].sum().sum() / g.vehicles.sum())
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(share.index, share.values * 100, marker="o", color="#3b6ea5")
ax.set_xlabel("Hour (JST)")
ax.set_ylabel("Large vehicles (%)")
ax.set_title("Share of large vehicles by hour")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Was it busier than usual?
Compare each day with the median of the same counter on the same weekday in the other weeks.
A ratio well above 1 marks an unusual day: holidays, detours, events.

In [ ]:
daily = (loop.assign(date=loop.time_jst.dt.normalize())
             .groupby(["counter_id", "date"])
             .agg(vehicles=("vehicles", "sum"), hours=("vehicles", "count"))
             .reset_index())
daily = daily[daily.hours == 24].drop(columns="hours")  # full days only
daily["weekday"] = daily.date.dt.dayofweek
usual = daily.groupby(["counter_id", "weekday"]).vehicles.transform("median")
n_same = daily.groupby(["counter_id", "weekday"]).vehicles.transform("size")
daily["ratio"] = np.where((n_same >= 5) & (usual > 0), daily.vehicles / usual, np.nan)

top = (daily.dropna(subset=["ratio"]).sort_values("ratio", ascending=False)
            .merge(counters[counters.sensor == "loop"][["counter_id", "prefecture", "municipality"]],
                   on="counter_id", how="left")
            .head(10))
top[["date", "counter_id", "prefecture", "municipality", "vehicles", "ratio"]]

In [ ]:
national = daily.groupby("date").ratio.median()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(national.index, national.values, color="#3b6ea5")
ax.axhline(1, color="grey", lw=1)
ax.set_ylabel("Median ratio to a usual day")
ax.set_title("Nationwide: traffic compared with the usual level for that weekday")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("Busiest days nationwide:", ", ".join(f"{d:%Y-%m-%d} ({r:.2f})" for d, r in national.nlargest(5).items()))

The peaks fall in holiday periods: the Marine Day weekend (2026-07-18 to 07-20), the week of
Mountain Day and Obon in August, and the five-day September break (Saturday 2026-09-19 to
Wednesday 09-23). A ratio compares a day with the same weekday, so a weekday holiday stands out
most.

## Notes
- Counts, not speeds: congestion has to be inferred.
- Values are reference values from the source, not official survey results.
- Source: 「交通量 API（国土交通省）機能による交通量(参考値)」を加工して作成
  （データ提供：公益財団法人日本道路交通情報センター）. Place names: 国土地理院.
- The archive grows every day: https://github.com/yasumorishima/japan-road-traffic